# Figure 5: Interpreting the black box
This notebook reproduces Figure 5A (t-SNE), 5B (attention heatmap), 5C (representative case), and 5D (error analysis).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Helvetica', 'Arial']
rcParams['axes.linewidth'] = 1.5
rcParams['xtick.major.width'] = 1.5
rcParams['ytick.major.width'] = 1.5

color_trauma = '#E64B35'
color_control = '#4DBBD5'
color_hr = '#D55E00'
color_bp = '#0072B2'
color_db = '#74B9FF'
color_risk = '#6C3483'

data_dir = './data'

In [ ]:
# Load data
df_tsne = pd.read_csv(f'{data_dir}/Fig5_tSNE.csv')

df_attn = pd.read_csv(f'{data_dir}/Fig5_Attention_Matrix.csv', index_col=0)
var_names = ['HR', 'SBP', 'DBP', 'SpO₂']
df_attn.index = var_names
df_attn.columns = var_names

df_case = pd.read_excel(f'{data_dir}/Fig4_CaseStudy.xlsx')
df_case['PP'] = df_case['SBP'] - df_case['DBP']
alert_time = df_case[df_case['Risk Probability'] >= 0.8]['Time min'].min()

df_error = pd.read_csv(f'{data_dir}/Fig5_ErrorAnalysis.csv')
df_fp = df_error[df_error['Case_Type'] == 'False Positive (Artifact)'].copy()
df_fn = df_error[df_error['Case_Type'] == 'False Negative (Compensation)'].copy()

In [ ]:
# Create figure
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1], height_ratios=[1, 1.2],
                      hspace=0.3, wspace=0.25)

# Panel A: t-SNE
ax1 = fig.add_subplot(gs[0, 0])
colors = {'Control': color_control, 'TIC': color_trauma}
for label, color in colors.items():
    subset = df_tsne[df_tsne['Label'] == label]
    ax1.scatter(subset['tSNE_dim1'], subset['tSNE_dim2'],
                c=color, label=label, s=20, alpha=0.7, edgecolors='none')
ax1.set_xlabel('t-SNE dimension 1', fontsize=11)
ax1.set_ylabel('t-SNE dimension 2', fontsize=11)
ax1.set_title('A  Latent space separation (t-SNE)', fontsize=12, loc='left', weight='bold')
ax1.legend(loc='upper right', fontsize=9, frameon=False)
ax1.grid(True, linestyle='--', alpha=0.5)

# Panel B: attention heatmap
ax2 = fig.add_subplot(gs[0, 1])
sns.heatmap(df_attn, annot=True, fmt='.2f', cmap='Reds',
            xticklabels=var_names, yticklabels=var_names,
            vmin=0, vmax=0.6, cbar=True, ax=ax2,
            annot_kws={'size': 10}, square=True)
# Highlight HR-BP interactions
for i, var in enumerate(var_names):
    if var == 'HR':
        for j, col in enumerate(var_names):
            if col in ['SBP', 'DBP']:
                ax2.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                                             edgecolor='yellow', lw=3))
ax2.set_title('B  Cross-variable attention weights\n(HR–BP interaction highlighted)',
              fontsize=12, loc='left', weight='bold')

# Panel C: representative case
ax3 = fig.add_subplot(gs[1, 0])
ax3.set_xlabel('Time (min)', fontsize=11)
ax3.set_ylabel('HR / BP (mmHg)', fontsize=11, color='black')
l1 = ax3.plot(df_case['Time min'], df_case['Heart Rate'], color=color_hr, linewidth=2, label='HR')
l2 = ax3.plot(df_case['Time min'], df_case['SBP'], color=color_bp, linewidth=2, label='SBP')
l3 = ax3.plot(df_case['Time min'], df_case['DBP'], color=color_db, linewidth=2, label='DBP')
ax3.fill_between(df_case['Time min'], df_case['DBP'], df_case['SBP'],
                 color=color_bp, alpha=0.15)
ax3_twin = ax3.twinx()
ax3_twin.set_ylabel('Risk Probability', fontsize=11, color=color_risk)
l4 = ax3_twin.plot(df_case['Time min'], df_case['Risk Probability'],
                   color=color_risk, linewidth=2.5, label='Risk')
if not np.isnan(alert_time):
    ax3.axvline(x=alert_time, color='gray', linestyle='--', linewidth=2, alpha=0.8)
    ax3.text(alert_time+0.5, 110, f'First alert\n({alert_time:.0f} min)',
             fontsize=10, color=color_trauma, weight='bold')
lines = l1 + l2 + l3 + l4
labels = [l.get_label() for l in lines]
ax3.legend(lines, labels, loc='upper left', fontsize=9, frameon=False)
ax3.set_xlim(0, 30)
ax3.set_ylim(50, 140)
ax3_twin.set_ylim(0, 1.05)
ax3.set_title('C  Representative case: early warning at 12 min', fontsize=12, loc='left', weight='bold')
ax3.grid(True, linestyle='--', alpha=0.5)

# Panel D: error analysis (two subplots)
gs_right = gs[1, 1].subgridspec(2, 1, hspace=0.3)
ax4_fp = fig.add_subplot(gs_right[0])
ax4_fn = fig.add_subplot(gs_right[1])

# False positive
ax4_fp.plot(df_fp['Time_min'], df_fp['Heart_Rate'], color=color_hr, linewidth=1.8, label='HR')
ax4_fp_twin = ax4_fp.twinx()
ax4_fp_twin.plot(df_fp['Time_min'], df_fp['Risk_Probability'],
                 color=color_risk, linewidth=2, linestyle='--', label='Risk')
ax4_fp.set_ylabel('HR (bpm)', fontsize=10)
ax4_fp_twin.set_ylabel('Risk', fontsize=10)
ax4_fp.set_title('False Positive (motion artifact)', fontsize=11, loc='left')
ax4_fp.set_xlim(0, 30)
ax4_fp.set_ylim(60, 110)
ax4_fp_twin.set_ylim(0, 0.5)
ax4_fp.grid(True, linestyle='--', alpha=0.3)

# False negative
ax4_fn.plot(df_fn['Time_min'], df_fn['Heart_Rate'], color=color_hr, linewidth=1.8, label='HR')
ax4_fn_twin = ax4_fn.twinx()
ax4_fn_twin.plot(df_fn['Time_min'], df_fn['Risk_Probability'],
                 color=color_risk, linewidth=2, linestyle='--', label='Risk')
ax4_fn.set_xlabel('Time (min)', fontsize=11)
ax4_fn.set_ylabel('HR (bpm)', fontsize=10)
ax4_fn_twin.set_ylabel('Risk', fontsize=10)
ax4_fn.set_title('False Negative (strong compensation)', fontsize=11, loc='left')
ax4_fn.set_xlim(0, 30)
ax4_fn.set_ylim(60, 110)
ax4_fn_twin.set_ylim(0, 0.5)
ax4_fn.grid(True, linestyle='--', alpha=0.3)

fig.text(0.75, 0.52, 'D  Error analysis', fontsize=12, weight='bold', ha='center')

plt.tight_layout()
plt.savefig('Figure5.tiff', dpi=300, format='tiff', bbox_inches='tight')
plt.show()